# Notebook 08 — Distortion and Undistortion
### 6D Pose Vision Workshop · Part 3: Camera Model

**Estimated time:** 40 minutes  
**Dependencies:** `opencv-contrib-python`, `numpy`, `matplotlib`

> **Grounded in:** video notes 3 (*Camera Calibration Explained*), 4 (*OpenCV Python Camera Calibration*)

---

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install opencv-contrib-python numpy matplotlib -q
    print("Running in Google Colab")
else:
    print("Running locally — make sure your venv is activated")

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os, json

print(f"OpenCV: {cv2.__version__}")

def show_images(pairs, figsize_per=(5, 4)):
    n = len(pairs)
    fig, axes = plt.subplots(1, n, figsize=(figsize_per[0]*n, figsize_per[1]))
    if n == 1: axes = [axes]
    for ax, (img, title) in zip(axes, pairs):
        ax.imshow(img[:,:,::-1] if img.ndim==3 else img, cmap=None if img.ndim==3 else 'gray')
        ax.set_title(title, fontsize=9); ax.axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
# Load calibration from previous notebook (or use defaults)
calib_path = '../assets/calibration/camera_calibration.npz'

if os.path.exists(calib_path):
    data = np.load(calib_path)
    K    = data['K']
    dist = data['dist']
    print(f"Loaded calibration from {calib_path}")
else:
    # Use plausible defaults if notebook 07 wasn't run
    K    = np.array([[720, 0, 320], [0, 720, 240], [0, 0, 1]], dtype=np.float64)
    dist = np.array([-0.3, 0.1, 0.001, 0.002, -0.05])
    print("Using default calibration values (run notebook 07 first for real values)")

print(f"K:\n{K}")
print(f"dist: {dist.flatten()}")

## Learning Objectives

By the end of this notebook you will:

- Understand what **radial** and **tangential** distortion are physically
- Know the **5 distortion coefficients** `[k1, k2, p1, p2, k3]` and what each affects
- Apply `cv2.undistort` for simple one-shot undistortion
- Use `cv2.remap` for **real-time** undistortion (precompute maps, then apply per frame)
- Know when to use `alpha=0` vs `alpha=1` in `getOptimalNewCameraMatrix`
- Validate undistortion results visually

---

## 1. Types of Lens Distortion

### Why real cameras distort

The pinhole camera model assumes a perfect lens — straight rays, no bending. Real lenses bend light, and this bending increases the further you get from the optical axis (center). This creates **distortion**.

### Radial distortion — the most common type

Points are displaced radially outward or inward from the image center:

$$x_{\text{distorted}} = x(1 + k_1 r^2 + k_2 r^4 + k_3 r^6)$$
$$y_{\text{distorted}} = y(1 + k_1 r^2 + k_2 r^4 + k_3 r^6)$$

Where $r^2 = x^2 + y^2$ is the squared distance from the image center.

**Annotation:**
- $k_1, k_2, k_3$ — radial distortion coefficients
- $r$ — normalized distance from center
- Negative $k_1$ → barrel distortion (pincushion if positive)

```
Barrel distortion (k1 < 0)       Pincushion distortion (k1 > 0)
  ┌───────────────┐               ┌──────────────────────────────┐
  │ ╔═══════════╗ │               │ ╔════╗                ╔════╗ │
  │ ║  bulges   ║ │               │ ║    ║                ║    ║ │
  │ ║  outward  ║ │               │ ║    ║    pinches     ║    ║ │
  │ ╚═══════════╝ │               │ ╚════╝    inward      ╚════╝ │
  └───────────────┘               └──────────────────────────────┘
  (wide-angle, fisheye lenses)    (telephoto, budget webcams)
```

### Tangential distortion — lens misalignment

Caused by the lens not being perfectly parallel to the image sensor:

$$x_{\text{distorted}} = x + [2p_1 xy + p_2(r^2 + 2x^2)]$$
$$y_{\text{distorted}} = y + [p_1(r^2 + 2y^2) + 2p_2 xy]$$

Where $p_1, p_2$ are the tangential distortion coefficients. Usually very small in modern cameras.

### OpenCV's distortion coefficient vector

$$\mathbf{d} = [k_1, \; k_2, \; p_1, \; p_2, \; k_3]$$

This is the `dist` array returned by `calibrateCamera`. For most cameras, $|k_3|$ and $|p_1|, |p_2|$ are very small.

In [ ]:
# Visualize what different distortion coefficients look like

def apply_radial_distortion(image, k1, k2=0, k3=0, K_mat=None):
    """Apply radial distortion to an image for visualization."""
    h, w = image.shape[:2]
    if K_mat is None:
        K_mat = np.array([[w*0.8, 0, w/2], [0, w*0.8, h/2], [0, 0, 1]], dtype=np.float64)
    dist_coeffs = np.array([k1, k2, 0, 0, k3])
    
    # Build distortion map
    map1, map2 = cv2.initUndistortRectifyMap(K_mat, dist_coeffs, None, K_mat, (w, h), cv2.CV_32FC1)
    # Note: initUndistortRectifyMap gives the UNDISTORT map
    # To APPLY distortion, we use the inverse — here we abuse it for visualization
    return cv2.remap(image, map1, map2, cv2.INTER_LINEAR)

# Create a grid image (straight lines show distortion clearly)
def create_grid_image(h=400, w=400, spacing=40):
    img = np.ones((h, w, 3), dtype=np.uint8) * 240
    for x in range(0, w, spacing):
        cv2.line(img, (x, 0), (x, h), (150, 150, 150), 1)
    for y in range(0, h, spacing):
        cv2.line(img, (0, y), (w, y), (150, 150, 150), 1)
    # Add a few colored reference points
    for x in range(spacing, w, spacing*2):
        for y in range(spacing, h, spacing*2):
            cv2.circle(img, (x, y), 4, (200, 50, 50), -1)
    return img

grid = create_grid_image()

K_vis = np.array([[320, 0, 200], [0, 320, 200], [0, 0, 1]], dtype=np.float64)

cases = [
    (grid,                                        'Original (no distortion)'),
    (apply_radial_distortion(grid, -0.5, 0.2, 0, K_vis), 'k1=-0.5 (barrel)'),
    (apply_radial_distortion(grid,  0.5, 0.0, 0, K_vis), 'k1=+0.5 (pincushion)'),
    (apply_radial_distortion(grid, -0.8, 0.5, 0, K_vis), 'k1=-0.8, k2=+0.5 (fisheye-like)'),
]

show_images(cases, figsize_per=(4, 4))
print("Grid lines should be straight — distortion bends them")

## 2. Undistortion Methods

### Method A: cv2.undistort (simple, best for single images)

```python
undistorted = cv2.undistort(img, K, dist, None, K_new)
```

Internally: recomputes the pixel mapping on every call. Fine for offline processing, slow for real-time.

### Method B: cv2.remap (fast, best for video)

Split into two steps:

```python
# Step 1: precompute maps ONCE before the video loop
mapx, mapy = cv2.initUndistortRectifyMap(K, dist, None, K_new, (w, h), cv2.CV_32FC1)

# Step 2: apply maps to every frame (very fast)
undistorted = cv2.remap(frame, mapx, mapy, cv2.INTER_LINEAR)
```

The maps are arrays of the same size as the image, storing for each output pixel where to sample from the distorted input. Computing them once and reusing is **3–5× faster** than `undistort` per frame.

### getOptimalNewCameraMatrix — controlling the crop

```python
K_new, roi = cv2.getOptimalNewCameraMatrix(K, dist, (w, h), alpha)
```

- `alpha=0` → crop to show only valid (undistorted) pixels — no black borders
- `alpha=1` → keep all pixels, including black border regions where distortion mapped outside
- `roi` → valid pixel region (use to crop if `alpha=0`)

In [ ]:
# Create a test image with strong distortion applied

# First, create a clean grid image
h, w = 480, 640
clean = create_grid_image(h, w, spacing=50)

# Apply strong barrel distortion (simulate a cheap wide-angle camera)
K_strong = np.array([[600, 0, w/2], [0, 600, h/2], [0, 0, 1]], dtype=np.float64)
dist_strong = np.array([-0.4, 0.15, 0.001, 0.002, -0.05])

# Distort the image (simulate what the camera captures)
map1, map2 = cv2.initUndistortRectifyMap(
    K_strong, dist_strong, None, K_strong, (w, h), cv2.CV_32FC1)
distorted = cv2.remap(clean, map1, map2, cv2.INTER_LINEAR)

# --- Method A: cv2.undistort ---
K_new_a, roi_a = cv2.getOptimalNewCameraMatrix(K_strong, dist_strong, (w, h), alpha=1)
undist_a = cv2.undistort(distorted, K_strong, dist_strong, None, K_new_a)

# --- Method B: cv2.remap (precomputed maps) ---
K_new_b, roi_b = cv2.getOptimalNewCameraMatrix(K_strong, dist_strong, (w, h), alpha=0)
mapx_b, mapy_b = cv2.initUndistortRectifyMap(
    K_strong, dist_strong, None, K_new_b, (w, h), cv2.CV_32FC1)
undist_b = cv2.remap(distorted, mapx_b, mapy_b, cv2.INTER_LINEAR)

# Show results
show_images([
    (clean,     'Original clean grid'),
    (distorted, 'Distorted (barrel k1=-0.4)'),
    (undist_a,  'undistort (alpha=1, full frame)'),
    (undist_b,  'remap (alpha=0, cropped to valid)'),
])

print(f"K_new (alpha=0): fx={K_new_b[0,0]:.1f}, cx={K_new_b[0,2]:.1f}")
print(f"Valid ROI (alpha=0): {roi_b}")

In [ ]:
# Compare alpha=0 vs alpha=1 behavior

results = []
for alpha in [0.0, 0.5, 1.0]:
    K_new_a, roi = cv2.getOptimalNewCameraMatrix(K_strong, dist_strong, (w, h), alpha)
    mapx_a, mapy_a = cv2.initUndistortRectifyMap(
        K_strong, dist_strong, None, K_new_a, (w, h), cv2.CV_32FC1)
    und = cv2.remap(distorted, mapx_a, mapy_a, cv2.INTER_LINEAR)
    results.append((und, f'alpha={alpha}\nroi={roi}'))

show_images(results)

print("alpha=0: all pixels valid, but image content is cropped")
print("alpha=1: full original field of view, black borders where distortion mapped outside")
print("\nFor robotics: use alpha=0 to avoid black border artifacts in pose estimation")

## 3. Real-Time Undistortion Loop

This is the pattern to use in all real-time robotics applications:

In [ ]:
# Real-time undistortion template (for use in your robotics code)

print("Real-time undistortion loop pattern:")
print("=" * 60)

template = '''
import cv2
import numpy as np

# --- SETUP: load calibration once at startup ---
data = np.load("camera_calibration.npz")
K, dist = data["K"], data["dist"]

cap = cv2.VideoCapture(0)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Precompute undistortion maps ONCE — this is the key for real-time speed
K_new, roi = cv2.getOptimalNewCameraMatrix(K, dist, (w, h), alpha=0)
mapx, mapy = cv2.initUndistortRectifyMap(
    K, dist, None, K_new, (w, h), cv2.CV_32FC1
)

print(f"Camera: {w}x{h}, K_new: fx={K_new[0,0]:.1f}, cx={K_new[0,2]:.1f}")

# --- LOOP: apply maps to every frame ---
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Apply undistortion (very fast — just a lookup table)
    frame_undistorted = cv2.remap(frame, mapx, mapy, cv2.INTER_LINEAR)

    # Continue with pose estimation, ArUco detection, etc.
    # Use K_new (not the original K) for solvePnP with undistorted images!
    # And use zero distortion (None or np.zeros(5)) since distortion is already removed.

    cv2.imshow("Undistorted", frame_undistorted)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
'''
print(template)

print("KEY POINT: After undistorting with K_new, use K_new and dist=zeros for pose estimation")

In [ ]:
# Benchmark: undistort vs remap performance

import time

# Create a test frame
test_frame = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)

# Precompute remap maps
K_new_r, _ = cv2.getOptimalNewCameraMatrix(K_strong, dist_strong, (640, 480), alpha=0)
mapx_r, mapy_r = cv2.initUndistortRectifyMap(
    K_strong, dist_strong, None, K_new_r, (640, 480), cv2.CV_32FC1)

N = 100

# Benchmark cv2.undistort
t0 = time.perf_counter()
for _ in range(N):
    _ = cv2.undistort(test_frame, K_strong, dist_strong, None, K_new_r)
t_undistort = (time.perf_counter() - t0) / N * 1000

# Benchmark cv2.remap (precomputed)
t0 = time.perf_counter()
for _ in range(N):
    _ = cv2.remap(test_frame, mapx_r, mapy_r, cv2.INTER_LINEAR)
t_remap = (time.perf_counter() - t0) / N * 1000

print(f"cv2.undistort: {t_undistort:.2f} ms/frame → {1000/t_undistort:.0f} FPS max")
print(f"cv2.remap:     {t_remap:.2f} ms/frame  → {1000/t_remap:.0f} FPS max")
print(f"Speedup: {t_undistort/t_remap:.1f}×")
print("\n→ Use remap for video — precompute once, apply every frame")

## 4. Validating Undistortion

### The straight-line test

The easiest visual validation: straight physical lines (walls, doors, table edges) should appear straight after undistortion. If they're still curved, recalibrate.

In [ ]:
# Overlay reprojected calibration points — quantitative validation

# Take one calibration image, undistort it, and check that projected corners
# land where findChessboardCorners detects them

import glob

def load_calibration_npz(path):
    d = np.load(path)
    return d['K'], d['dist']

# Try to load from file, fallback to strong distortion example
K_val, dist_val = (K_strong, dist_strong)

# Create a grid image distorted by our model
orig = create_grid_image(480, 640, spacing=40)

# Add horizontal reference line (should be straight after undistortion)
cv2.line(orig, (0, 240), (640, 240), (0, 0, 200), 2)
cv2.line(orig, (320, 0), (320, 480), (0, 0, 200), 2)
cv2.putText(orig, 'These lines must be straight after undistort',
            (10, 460), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 200), 1)

# Distort then undistort
map_d1, map_d2 = cv2.initUndistortRectifyMap(
    K_val, dist_val, None, K_val, (640, 480), cv2.CV_32FC1)
distorted_val = cv2.remap(orig, map_d1, map_d2, cv2.INTER_LINEAR)

K_new_v, _ = cv2.getOptimalNewCameraMatrix(K_val, dist_val, (640, 480), alpha=0)
mapx_v, mapy_v = cv2.initUndistortRectifyMap(
    K_val, dist_val, None, K_new_v, (640, 480), cv2.CV_32FC1)
undistorted_val = cv2.remap(distorted_val, mapx_v, mapy_v, cv2.INTER_LINEAR)

show_images([
    (orig,            'Original (perfect grid)'),
    (distorted_val,   'After barrel distortion (lines curve)'),
    (undistorted_val, 'After undistortion (lines straight again)'),
])

print("If lines are straight again → undistortion is working correctly")
print("If lines are still curved → check that K and dist match the actual camera")

## Recap

| Concept | Key point |
|---|---|
| Radial distortion | $k_1, k_2, k_3$ — barrel (k1<0) or pincushion (k1>0) |
| Tangential distortion | $p_1, p_2$ — lens/sensor misalignment; usually small |
| dist vector | `[k1, k2, p1, p2, k3]` — OpenCV format |
| `undistort` | Simple; recomputes map every call; use for single images |
| `remap` | Fast; precompute with `initUndistortRectifyMap`; use for video |
| alpha=0 | Crop to valid pixels only (no black borders) |
| alpha=1 | Keep full FOV, include black border areas |
| After undistortion | Use `K_new` (not original K) and `dist=zeros` for pose estimation |

---

**Next:** `09_solvePnP_explained.ipynb` — now that we have K and can remove distortion, use corresponding 3D-2D point pairs to estimate the full 6D pose.

## Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Effect of each distortion coefficient
# ============================================================
# Create a grid image and apply distortion with ONLY ONE coefficient nonzero:
#   1. k1 only (try -0.5 and +0.5)
#   2. k2 only (try +0.2)
#   3. p1 only (try +0.1)
#   4. p2 only (try +0.1)
# Display all 5 images and describe what each coefficient does.

# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# grid = create_grid_image(400, 400, spacing=40)
# K_ex = np.array([[350, 0, 200], [0, 350, 200], [0, 0, 1]], dtype=np.float64)
# cases = []
# for name, d_arr in [
#     ('Original',    [0, 0, 0, 0, 0]),
#     ('k1=-0.5',     [-0.5, 0, 0, 0, 0]),
#     ('k1=+0.5',     [0.5, 0, 0, 0, 0]),
#     ('k2=+0.2',     [0, 0.2, 0, 0, 0]),
#     ('p1=+0.15',    [0, 0, 0.15, 0, 0]),
#     ('p2=+0.15',    [0, 0, 0, 0.15, 0]),
# ]:
#     d = np.array(d_arr)
#     if np.all(d == 0):
#         cases.append((grid, name))
#     else:
#         m1, m2 = cv2.initUndistortRectifyMap(K_ex, d, None, K_ex, (400,400), cv2.CV_32FC1)
#         cases.append((cv2.remap(grid, m1, m2, cv2.INTER_LINEAR), name))
#
# show_images(cases, figsize_per=(3, 3))
# # k1<0 → barrel (lines curve outward)
# # k1>0 → pincushion (lines curve inward)
# # k2 → adds higher-order radial correction
# # p1/p2 → diagonal shift (lines shift asymmetrically)

In [ ]:
# ============================================================
# EXERCISE 2: Build a complete undistortion function
# ============================================================
# Write undistort_frame(frame, K, dist, alpha=0) that:
#   1. Computes K_new with getOptimalNewCameraMatrix
#   2. Precomputes maps with initUndistortRectifyMap
#   3. Applies remap
#   4. Returns (undistorted_frame, K_new)
#
# Note: in a real video loop, precompute maps OUTSIDE the loop.
# This exercise is for the single-frame case.

# YOUR CODE HERE
def undistort_frame(frame, K, dist, alpha=0):
    pass


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# def undistort_frame(frame, K, dist, alpha=0):
#     h, w = frame.shape[:2]
#     K_new, roi = cv2.getOptimalNewCameraMatrix(K, dist, (w, h), alpha)
#     mapx, mapy = cv2.initUndistortRectifyMap(K, dist, None, K_new, (w, h), cv2.CV_32FC1)
#     undistorted = cv2.remap(frame, mapx, mapy, cv2.INTER_LINEAR)
#     return undistorted, K_new
#
# # Test
# test_frame = distorted_val.copy()
# result, K_new_out = undistort_frame(test_frame, K_val, dist_val, alpha=0)
# print(f"Output shape: {result.shape}")
# print(f"K_new: fx={K_new_out[0,0]:.1f}, cx={K_new_out[0,2]:.1f}")
# show_images([(test_frame, 'Distorted'), (result, 'Undistorted')])